# Tournament Simulation

This notebook will explore tournament workflows, group stages, knockout progression, and Monte Carlo simulations.

In [ ]:
import pandas as pd
import pickle
with open("../models/fifa_wc_model.pkl", "rb") as f:
    model = pickle.load(f)
with open("../models/feature_columns.pkl", "rb") as f:
    feature_columns = pickle.load(f)
team_features = pd.read_csv(
    "../datasets/processed/team_features.csv"
)

In [ ]:
team_lookup = (
    team_features
    .set_index("nationality")
    .to_dict("index")
)

In [ ]:
import pandas as pd

def predict_match(team1, team2):

    # Check teams exist
    if team1 not in team_lookup:
        raise ValueError(f"{team1} not found in team_features")

    if team2 not in team_lookup:
        raise ValueError(f"{team2} not found in team_features")

    home = team_lookup[team1]
    away = team_lookup[team2]

    # Create feature vector
    features = {

        "attack_diff":
            home["attack"] -
            away["attack"],

        "midfield_diff":
            home["midfield"] -
            away["midfield"],

        "defense_diff":
            home["defense"] -
            away["defense"],

        "goalkeeper_diff":
            home["goalkeeper"] -
            away["goalkeeper"],

        "overall_strength_diff":
            home["overall_strength"] -
            away["overall_strength"],

        "squad_depth_diff":
            home["squad_depth"] -
            away["squad_depth"],

        "superstar_diff":
            home["superstar_index"] -
            away["superstar_index"],

        "final_team_rating_diff":
            home["final_team_rating"] -
            away["final_team_rating"],

        "wins_last5_overall_diff":
            home["wins_last5_overall"] -
            away["wins_last5_overall"],

        "goals_scored_last5_diff":
            home["goals_scored_last5_overall"] -
            away["goals_scored_last5_overall"],

        "goals_conceded_last5_diff":
            home["goals_conceded_last5_overall"] -
            away["goals_conceded_last5_overall"],

        "elo_diff":
            home["elo"] -
            away["elo"],

        "elo_normalized_diff":
            home["elo_normalized"] -
            away["elo_normalized"],

        "enhanced_team_rating_diff":
            home["enhanced_team_rating"] -
            away["enhanced_team_rating"],

        # Uncomment ONLY if model was trained with it
        # ,"home_advantage": 0
    }

    # Convert to dataframe
    X_pred = pd.DataFrame([features])

    # Match training feature order
    X_pred = X_pred[feature_columns]

    # Debug output
    print("\nFeature Vector:\n")
    print(X_pred.T)

    # Predict probabilities
    probabilities = model.predict_proba(X_pred)[0]

    away_win = float(probabilities[0])
    draw = float(probabilities[1])
    home_win = float(probabilities[2])

    print("\nRaw Probabilities:")
    print(probabilities)

    print("\nClasses:")
    print(model.classes_)

    print("\n" + "=" * 45)
    print(f"{team1} vs {team2}")
    print("=" * 45)

    print(f"{team2} Win Probability : {away_win:.2f}")
    print(f"Draw Probability        : {draw:.2f}")
    print(f"{team1} Win Probability : {home_win:.2f}")

    print("=" * 45)

    predicted = max(
        [
            (f"{team2} Win", away_win),
            ("Draw", draw),
            (f"{team1} Win", home_win)
        ],
        key=lambda x: x[1]
    )

    print(
        f"Most Likely Outcome: "
        f"{predicted[0]} ({predicted[1]:.2f}%)"
    )

    return {
        "away_win": away_win,
        "draw": draw,
        "home_win": home_win
    }

In [ ]:
import numpy as np
def simulate_match(team1, team2):

    probs = predict_match(
        team1,
        team2
    )

    outcomes = [
        "away_win",
        "draw",
        "home_win"
    ]

    probabilities = [
        probs["away_win"],
        probs["draw"],
        probs["home_win"]
    ]

    probabilities = np.array(probabilities)

    probabilities = probabilities / probabilities.sum()

    result = np.random.choice(
        outcomes,
        p=probabilities
    )

    if result == "home_win":

        winner = team1

    elif result == "away_win":

        winner = team2

    else:

        winner = "Draw"

    return {
        "team1": team1,
        "team2": team2,
        "result": result,
        "winner": winner,
        "probabilities": probs
    }

In [ ]:
simulate_match(
    "Argentina",
    "France"
)

In [ ]:
predict_match("Argentina", "India")

In [ ]:
print(model.classes_)

In [ ]:
predict_match("England", "Spain")